# Corrigé du TP2 : nettoyage d'un jeu de données

## Objectifs

- sélectionner des lignes et des colonnes avec `loc`, `iloc`, des filtres booléens et `query`
- créer et transformer des colonnes : calculs vectorisés, `np.where`, `assign`, `map`, `apply`
- nettoyer des colonnes texte avec l'accesseur `.str`
- corriger les types : nombres, entiers nullables, dates, catégories
- traiter les valeurs manquantes, les valeurs aberrantes et les doublons
- trier, renommer les colonnes, gérer l'index
- appliquer une démarche complète de nettoyage au jeu de données des ventes, puis l'exporter en CSV et en Parquet

## Livrable

- un jeu de données propre, exporté en CSV et en Parquet, avec une comparaison de la taille des deux fichiers
- une courte note, en fin de notebook, listant les choix de nettoyage effectués

## Comment travailler

- Le TP suit le cours : chaque partie se fait juste après la section du cours correspondante. Les parties 1 à 6 sont des exercices courts, la partie 7 est le cas pratique : le nettoyage complet des ventes du TP1.
- Complète les cellules marquées `# TODO`, en utilisant les noms de variables indiqués dans l'énoncé.
- Quand un **résultat attendu** est indiqué, compare-le avec ce que tu obtiens.
- Les cellules qui commencent par `# Vérification` contrôlent ton travail avec `assert` : exécute-les sans les modifier. Si tout est juste, elles affichent `OK` ; sinon, elles s'arrêtent sur une `AssertionError` dont le message indique ce qui ne va pas.
- Les questions **Pour aller plus loin** sont facultatives : fais-les si tu as terminé une partie en avance.

## Partie 0 : environnement

Deux possibilités :

- réutiliser l'environnement du TP1 : depuis le dossier de ce TP, active-le avec `source ../tp01-environnement-pandas/.venv/bin/activate`, puis lance `jupyter lab` ;
- ou recréer un environnement dans ce dossier à partir du `requirements.txt` du TP1 :
  ```bash
  python3 -m venv .venv
  source .venv/bin/activate
  python -m pip install -r ../tp01-environnement-pandas/requirements.txt
  jupyter lab
  ```

Le fichier `ressources/ventes.csv` est le fichier brut : ne le modifie jamais. Toutes les corrections se font sur une copie en mémoire, et le résultat est écrit dans de nouveaux fichiers.

In [2]:
import numpy as np
import pandas as pd

brut = pd.read_csv("ressources/ventes.csv")
df = brut.copy()
df.head()


,id_commande,date,region,categorie,produit,Prix unitaire,quantite,Client fidèle
0,1021,2025-01-01,Nouvelle-Aquitaine,Bureau,Lampe LED,268.78,9,False
1,1035,2025-01-02,Bretagne,Bureau,Rangement papier,316.64,3,False
2,1044,2025-01-05,Hauts-de-France,Électroménager,Grille-pain,248.61,10,False
3,1026,2025-01-11,Occitanie,Électroménager,Cafetière,92.78,2,False
4,1053,2025-01-13,Bretagne,Bureau,Bureau assis-debout,341.86,3,True


## Partie 1 : sélectionner des données

*À faire après la section 1 du cours : sélectionner des données.*

On travaille sur `df`, la copie du fichier brut, avec ses noms de colonnes d'origine.

### Question 1.1

Affiche les colonnes `produit` et `quantite` des lignes de labels 10 à 14 avec `loc`. Obtiens ensuite exactement le même résultat avec `iloc`. Pourquoi les bornes ne sont-elles pas les mêmes ?

**Résultat attendu** :

```text
                produit  quantite
10          Stylo plume        10
11      Souris sans fil        10
12  Bureau assis-debout         6
13          Stylo plume         5
14          Grille-pain         9
                produit  quantite
10          Stylo plume        10
11      Souris sans fil        10
12  Bureau assis-debout         6
13          Stylo plume         5
14          Grille-pain         9
```

In [3]:
display(df.loc[10:14, ["produit", "quantite"]])
df.iloc[10:15, [4, 6]]


,produit,quantite
10,Stylo plume,10
11,Souris sans fil,10
12,Bureau assis-debout,6
13,Stylo plume,5
14,Grille-pain,9


,produit,quantite
10,Stylo plume,10
11,Souris sans fil,10
12,Bureau assis-debout,6
13,Stylo plume,5
14,Grille-pain,9


**Réponse** : `loc` sélectionne par label et inclut la borne de fin : `10:14` donne 5 lignes. `iloc` sélectionne par position et exclut la borne de fin : il faut écrire `10:15`. Les colonnes, elles, se désignent par leur nom avec `loc` et par leur position avec `iloc` (`produit` est en position 4, `quantite` en position 6).

### Question 1.2

Sélectionne dans `bureau_5` les commandes de la catégorie `Bureau` dont la quantité est supérieure ou égale à 5. Affiche leur nombre, puis leurs colonnes `id_commande`, `produit` et `quantite`.

**Résultat attendu** :

```text
11
    id_commande              produit  quantite
0          1021            Lampe LED         9
9          1027  Bureau assis-debout        12
12         1005  Bureau assis-debout         6
19         1050   Chaise ergonomique        10
22         1029     Rangement papier         8
38         1041            Lampe LED         7
39         1040   Chaise ergonomique         7
40         1040   Chaise ergonomique         7
47         1025            Lampe LED        11
55         1014  Bureau assis-debout         9
61         1020   Chaise ergonomique         5
```

In [4]:
bureau_5 = df[(df["categorie"] == "Bureau") & (df["quantite"] >= 5)]
print(len(bureau_5))
bureau_5[["id_commande", "produit", "quantite"]]


11


,id_commande,produit,quantite
0,1021,Lampe LED,9
9,1027,Bureau assis-debout,12
12,1005,Bureau assis-debout,6
19,1050,Chaise ergonomique,10
22,1029,Rangement papier,8
38,1041,Lampe LED,7
39,1040,Chaise ergonomique,7
40,1040,Chaise ergonomique,7
47,1025,Lampe LED,11
55,1014,Bureau assis-debout,9


In [5]:
# Vérification
assert len(bureau_5) == 11, "11 commandes attendues : vérifie les deux conditions et les parenthèses"
assert (bureau_5["categorie"] == "Bureau").all() and (bureau_5["quantite"] >= 5).all()
print("Question 1.2 : OK")


Question 1.2 : OK


### Question 1.3

Compte les commandes passées en Bretagne ou en Occitanie avec `isin`. Compare avec ce qu'indique `value_counts` sur la colonne `region` : que manque-t-il au filtre, et pourquoi ?

**Résultat attendu** :

```text
Bretagne ou Occitanie : 12
region
Hauts-de-France         18
Nouvelle-Aquitaine      10
Île-de-France            9
Occitanie                8
Auvergne-Rhône-Alpes     7
Bretagne                 4
NaN                      3
Bretagne                 1
occitanie                1
Ile-de-France            1
Name: count, dtype: int64
```

In [6]:
print("Bretagne ou Occitanie :", df["region"].isin(["Bretagne", "Occitanie"]).sum())
df["region"].value_counts(dropna=False)


Bretagne ou Occitanie : 12


region
Hauts-de-France         18
Nouvelle-Aquitaine      10
Île-de-France            9
Occitanie                8
Auvergne-Rhône-Alpes     7
Bretagne                 4
NaN                      3
Bretagne                 1
occitanie                1
Ile-de-France            1
Name: count, dtype: int64

**Réponse** : Le filtre trouve 12 commandes (4 en Bretagne, 8 en Occitanie). Il manque `Bretagne ` avec une espace finale et `occitanie` en minuscules : `isin` compare des valeurs exactes. Il faudra nettoyer le texte avant de filtrer.

### Question 1.4

En une seule instruction avec `loc`, affiche la date, le produit et la quantité des commandes de plus de 10 unités, triées de la plus grosse à la plus petite.

**Résultat attendu** :

```text
          date              produit  quantite
24  2025-04-18      Écran 24 pouces       480
9   2025-01-31  Bureau assis-debout        12
34  2025-05-15            Cafetière        12
36  2025-05-17            Agrafeuse        12
59  2025-09-07            Agrafeuse        12
26  2025-04-23           Bouilloire        11
44  2025-07-05            Cafetière        11
47  2025-07-11            Lampe LED        11
50  2025-08-03         Casque audio        11
```

In [7]:
df.loc[df["quantite"] > 10, ["date", "produit", "quantite"]].sort_values("quantite", ascending=False)


,date,produit,quantite
24,2025-04-18,Écran 24 pouces,480
9,2025-01-31,Bureau assis-debout,12
34,2025-05-15,Cafetière,12
36,2025-05-17,Agrafeuse,12
59,2025-09-07,Agrafeuse,12
26,2025-04-23,Bouilloire,11
44,2025-07-05,Cafetière,11
47,2025-07-11,Lampe LED,11
50,2025-08-03,Casque audio,11


### Question 1.5

Combien de commandes de clients non fidèles ont un prix unitaire compris entre 100 et 200 euros, bornes incluses ? Utilise `~` et `between`, et range le nombre dans `nb_commandes`.

**Résultat attendu** :

```text
10
```

In [8]:
nb_commandes = (~df["Client fidèle"] & df["Prix unitaire"].between(100, 200)).sum()
print(nb_commandes)


10


### Question 1.6

Avec `query`, sélectionne les commandes de clients fidèles d'au moins 10 unités, et affiche leurs colonnes `id_commande`, `produit` et `quantite`. Attention : le nom de colonne `Client fidèle` contient une espace et doit être entouré d'accents graves.

**Résultat attendu** :

```text
    id_commande              produit  quantite
9          1027  Bureau assis-debout        12
36         1054            Agrafeuse        12
50         1003         Casque audio        11
59         1036            Agrafeuse        12
```

In [ ]:
df.query("`Client fidèle` and quantite >= 10")[["id_commande", "produit", "quantite"]]


,id_commande,produit,quantite
9,1027,Bureau assis-debout,12
36,1054,Agrafeuse,12
50,1003,Casque audio,11
59,1036,Agrafeuse,12


### Question 1.7

Crée une copie `essai` de `df` avec `.copy()`. Dans `essai`, remplace par 0 les quantités négatives, en une seule instruction avec `loc`. Affiche ensuite le nombre de quantités négatives dans `essai`, puis dans `df`.

**Résultat attendu** :

```text
0 1
```

In [10]:
essai = df.copy()
essai.loc[essai["quantite"] < 0, "quantite"] = 0
print((essai["quantite"] < 0).sum(), (df["quantite"] < 0).sum())


0 1


In [11]:
# Vérification
assert (essai["quantite"] < 0).sum() == 0, "il reste des quantités négatives dans essai"
assert (df["quantite"] < 0).sum() == 1, "df ne doit pas être modifié : travaille sur la copie essai"
assert len(essai) == len(df), "aucune ligne ne doit être supprimée"
print("Question 1.7 : OK")


Question 1.7 : OK


### Pour aller plus loin 1.8

Exécute l'instruction `essai[essai["quantite"] > 100]["quantite"] = 0`, puis affiche la quantité maximale de `essai`. La quantité de 480 a-t-elle été modifiée ? Écris ensuite la bonne version.

In [12]:
essai[essai["quantite"] > 100]["quantite"] = 0
print(essai["quantite"].max())

essai.loc[essai["quantite"] > 100, "quantite"] = 0
print(essai["quantite"].max())


480
12


/var/folders/z6/kz74j4r96llbbc63nsb2zqsh0000gn/T/ipykernel_76702/3167100436.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  essai[essai["quantite"] > 100]["quantite"] = 0


**Réponse** : Non : la première sélection crée une copie temporaire, et l'affectation modifie cette copie, aussitôt perdue. pandas l'annonce par un avertissement `ChainedAssignmentError`. Pour modifier le DataFrame, on sélectionne et on affecte en une seule étape avec `loc`.